In [50]:
import torch as T
import torch.nn as nn

In [51]:
class CovtypeClassifier(nn.Module):
    def __init__(self,  input_size=54, num_classes=7):
        super().__init__()
        self.layer1 = nn.Linear(input_size, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.act1 = nn.ReLU()
        self.drop1 = nn.Dropout(p=0.3)
        self.layer2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.act2 = nn.ReLU()
        self.drop2 = nn.Dropout(p=0.3)
        self.layer3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.act3 = nn.ReLU()
        self.drop3 = nn.Dropout(0.2)

        self.output = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.act1(self.bn1(self.layer1(x)))
        x = self.drop1(x)
        x = self.act2(self.bn2(self.layer2(x)))
        x = self.drop2(x)
        x = self.act3(self.bn3(self.layer3(x)))
        x = self.drop3(x)

        x = self.output(x)
        return x

In [52]:
from sklearn.datasets import fetch_covtype
#num_classes 1-7
covtype = fetch_covtype()
X = covtype.data
y = covtype.target
y -=1


In [53]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.32, shuffle=True, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

In [54]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

In [55]:
from torch.utils.data import TensorDataset, DataLoader

X_train_t = T.tensor(X_train).float()
X_test_t = T.tensor(X_test).float()
X_val_t = T.tensor(X_val).float()
y_train_t = T.tensor(y_train).long()
y_test_t = T.tensor(y_test).long()
y_val_t = T.tensor(y_val).long()

In [56]:
train_dataset = TensorDataset(X_train_t, y_train_t)
test_dataset = TensorDataset(X_test_t, y_test_t)
val_dataset= TensorDataset(X_val_t, y_val_t)

In [57]:
batch_size=512

In [58]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

for inputs, labels in train_loader:
    # Ispituje tip podataka i dimenzije inputs i labels
    print(f"Tip podataka za inputs: {inputs.dtype}")
    print(f"Dimenzije inputs tenzora: {inputs.shape}")

    print(f"Tip podataka za labels: {labels.dtype}")
    print(f"Dimenzije labels tenzora: {labels.shape}")
    break


Tip podataka za inputs: torch.float32
Dimenzije inputs tenzora: torch.Size([512, 54])
Tip podataka za labels: torch.int64
Dimenzije labels tenzora: torch.Size([512])


In [59]:
device = T.device("cuda:0") if T.cuda.is_available() else T.device("cpu") # koristimo gpu samo ako je dostupan, inace koristimo cpu
print(device)

cuda:0


In [60]:
net = CovtypeClassifier().to(device)

In [61]:
# metoda za testiranje mreže
# loss_fn predstavlja funkciju koja računa grešku
from sklearn.metrics import recall_score, precision_score

def test_model(net, loss_fn, test_loader):
    total_loss = 0
    net.eval() # prebacujemo mrežu u mod za testiranje/evaluaciju
    ground_truth_labels = []  # Lista za prave oznake (stvarne klase)
    predicted_labels = []  # Lista za predikcije neuronske mreže

    with T.no_grad(): # ne treba nam računanje gradijenta jer radimo u modu testiranja
        for x, y in test_loader: # iteriramo kroz dataloader, dobijamo podatke u batchu od 32
            x = x.to(device) # potrebno i x i y prebaciti na odgovarajući uređaj
            y = y.to(device)
            preds = net(x) # generišemo predikcije
            loss = loss_fn(preds, y) # računamo grešku
            total_loss += loss.item() # jako bitno pozvati .item() funkciju ako nam treba samo broj,
                                      #jer loss čuva referencu na generisani graf izračunavanja koji nam nije potreban u ovom slučaju
            ground_truth_labels.extend(y.cpu().numpy())
            predicted_labels.extend(preds.argmax(dim=1).cpu().numpy())

    precision = precision_score(ground_truth_labels, predicted_labels, average='weighted', zero_division=0.0)
    recall = recall_score(ground_truth_labels, predicted_labels, average='weighted', zero_division=0.0)

    # Ispisivanje rezultata
    print("Average Loss:", total_loss / len(test_loader))
    print("Preciznost (Precision):", precision)
    print("Odziv (Recall):", recall)

In [62]:
# optimizer klasa čuva parametre mreže i omogućava njihovu promjenu na osnovu izračunatih gradijenata
# weight_decay opcija implementira regularizaciju
# Adam adaptivno mijenja parametar učenja tokom treninga što mu često daje bolje performanse od standardnog gradijetnog spusta
optimizer = T.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5)
#  Definisemo adekvatnu funkciju greske za problem
loss_fn = nn.CrossEntropyLoss()

In [63]:
test_model(net, loss_fn, test_loader)


Average Loss: 1.9453452013351105
Preciznost (Precision): 0.2542898432052571
Odziv (Recall): 0.051322045567005876


In [65]:
EPOCHS = 50
scheduler = T.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=3
)

In [66]:
test_model(net, loss_fn, test_loader)

Average Loss: 1.9453452013351105
Preciznost (Precision): 0.2542898432052571
Odziv (Recall): 0.051322045567005876


In [67]:
net.train() # stavljamo mrežu u mod rada za trening
max_patience = 5  # Maximum number of epochs to tolerate rising validation loss
patience_counter = 0  # Counter for consecutive epochs with rising validation loss
best_val_loss = float('inf')  # Initialize best validation loss to a high value
best_val_acc = 0.0

train_losses =[]
val_losses = []

for i in range(EPOCHS):
    if i % 2 == 0:
        print(f"Running epoch: {i}")
    net.train()
    epoch_train_loss = 0.0
    # slicni koraci kao kod testiranja
    # ovaj put ne koristimo T.no_grad jer nam trebaju gradijenti
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        # nuliramo prethodno izračunate gradijente
        optimizer.zero_grad()
        preds = net(x)
        loss = loss_fn(preds, y)
        # prograpigramo grešku unazad
        loss.backward()

        T.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)
        # pravimo korak - mijenjamo parametre mreže na osnovu gradijenta i faktora obučavanja
        optimizer.step()
        epoch_train_loss += loss.item()
    avg_train_loss = epoch_train_loss / len(train_loader)

    net.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with T.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            output = net(batch_X)
            val_loss += loss_fn(output, batch_y).item()
            _, predicted = T.max(output.data, 1)
            total += batch_y.size(0)
            correct += (predicted ==batch_y).sum().item()

    avg_val_loss =  val_loss / len(val_loader)
    val_accuracy = correct / total

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)

    scheduler.step(avg_val_loss)

    print(f"Epoch [{i+1}/{EPOCHS}] - "
          f"Train Loss: {avg_train_loss:.4f} - "
          f"Val Loss: {avg_val_loss:.4f} - "
          f"Val Acc: {val_accuracy:.4f}")

     # Early stopping logic
    if val_accuracy > best_val_acc:
        best_val_acc = val_accuracy
        best_val_loss = avg_val_loss
        patience_counter = 0
            # Save best model
        T.save(net.state_dict(), 'best_model.pth')
    else:
        patience_counter += 1

    if patience_counter >= max_patience:
        print(f"Early stopping triggered after {i+1} epochs.")
        break
##dodato novo
net.load_state_dict(T.load('best_model.pth'))
test_model(net, loss_fn, test_loader)


Running epoch: 0
Epoch [1/50] - Train Loss: 0.6141 - Val Loss: 0.4939 - Val Acc: 0.7856
Epoch [2/50] - Train Loss: 0.5069 - Val Loss: 0.4355 - Val Acc: 0.8145
Running epoch: 2
Epoch [3/50] - Train Loss: 0.4686 - Val Loss: 0.4035 - Val Acc: 0.8300
Epoch [4/50] - Train Loss: 0.4424 - Val Loss: 0.3773 - Val Acc: 0.8392
Running epoch: 4
Epoch [5/50] - Train Loss: 0.4237 - Val Loss: 0.3539 - Val Acc: 0.8521
Epoch [6/50] - Train Loss: 0.3986 - Val Loss: 0.3313 - Val Acc: 0.8625
Running epoch: 6
Epoch [7/50] - Train Loss: 0.3865 - Val Loss: 0.3186 - Val Acc: 0.8678
Epoch [8/50] - Train Loss: 0.3778 - Val Loss: 0.3107 - Val Acc: 0.8714
Running epoch: 8
Epoch [9/50] - Train Loss: 0.3712 - Val Loss: 0.3013 - Val Acc: 0.8767
Epoch [10/50] - Train Loss: 0.3588 - Val Loss: 0.2894 - Val Acc: 0.8811
Running epoch: 10
Epoch [11/50] - Train Loss: 0.3536 - Val Loss: 0.2859 - Val Acc: 0.8826
Epoch [12/50] - Train Loss: 0.3501 - Val Loss: 0.2815 - Val Acc: 0.8847
Running epoch: 12
Epoch [13/50] - Train Lo

In [68]:
test_model(net, loss_fn, test_loader)

Average Loss: 0.2519782576914672
Preciznost (Precision): 0.8983059018885677
Odziv (Recall): 0.8983885888857813
